In [35]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
from IPython.core.debugger import set_trace

# Definition of parameters

In [36]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
learning_rate = 0.00001
batch_size = 50
experiment_name = 'prova1'

Using device: mps


In [37]:
import torch
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

MPS built: True
MPS available: True


# Data loading

In [38]:
class Dataset(torch.utils.data.Dataset):

    def __init__(self, csv):
        # read the csv file
        self.df = pd.read_csv(csv, sep='\s+')
        # self.df = self.df.dropna(axis=0)
        # save cols
        self.input_cols = ['pickup_longitude',    'pickup_latitude',    'dropoff_longitude',    'dropoff_latitude',   'passenger_count',      
                            'year',      'Distance',   'month_1',    'month_2',    'month_3',    'month_4',    'month_5',    'month_6',    'month_7',    
                            'month_8',    'month_9',    'month_10',    'month_11',    'month_12',    'weekday_0',    'weekday_1',    'weekday_2',    
                            'weekday_3',    'weekday_4',    'weekday_5',    'weekday_6',     'hour_0',     'hour_1',     'hour_2',     'hour_3',     'hour_4',     
                              'hour_5',     'hour_6',     'hour_7',     'hour_8',     'hour_9',    'hour_10',    'hour_11',    'hour_12',    'hour_13',    
                              'hour_14',    'hour_15',    'hour_16',    'hour_17',    'hour_18',    'hour_19',    'hour_20',    'hour_21',    'hour_22',    'hour_23']
        self.output_cols = ['fare_amount']
        


    def __len__(self):
        # here i will return the number of samples in the dataset
        return len(self.df)


    def __getitem__(self, idx):
        # here i will load the file in position idx
        cur_sample = self.df.iloc[idx]
        if cur_sample.isna().any():
            set_trace()()
        # split in input / ground-truth
        cur_sample_x = cur_sample[self.input_cols]
        cur_sample_y = cur_sample[self.output_cols]
        # convert to torch format
        cur_sample_x = torch.tensor(cur_sample_x.tolist())
        cur_sample_y = torch.tensor(cur_sample_y.tolist())
        # return values
        return cur_sample_x, cur_sample_y


<>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/ly/1dlrl_n54xx769z_pqq07nkr0000gn/T/ipykernel_1078/173937811.py:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  self.df = pd.read_csv(csv, sep='\s+')


In [39]:
# try to use the dataset
ds = Dataset('../dataset/Uber/train.csv')
# get first item
xx,yy = ds.__getitem__(0)
# print shapes
print(xx.shape)
print(yy.shape)

torch.Size([50])
torch.Size([1])


In [40]:
# create train and validation datasets
train_ds = Dataset('../dataset/Uber/train.csv')
val_ds =  Dataset('../dataset/Uber/val.csv')

In [57]:
from torch.utils.data import DataLoader

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_dl = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# Network definition

In [51]:
# define network

class Net(nn.Module):

    def __init__(self):
        # initialize super class
        super(Net, self).__init__()
        self.layer1 = nn.Linear(13,128)
        self.layer2 = nn.ReLU()
        self.layer3 = nn.Linear(128,64)
        self.layer4 = nn.ReLU()
        self.layer5 = nn.Linear(64,32)
        self.layer6 = nn.ReLU()
        self.layer7 = nn.Linear(32,16)
        self.layer8 = nn.ReLU()
        self.layer9 = nn.Linear(16, 1)


    def forward(self, x):
        # apply layers in cascade
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.layer7(x)
        x = self.layer8(x)
        x = self.layer9(x)
        # return output
        return x


In [52]:
# let's test the network
net = Net()

# define random batch of 10 elements
inp = torch.rand(10, 13)

# forward
out = net(inp)

# let's print the shape
print(' Input shape is', inp.shape)
print('Output shape is', out.shape)

 Input shape is torch.Size([10, 13])
Output shape is torch.Size([10, 1])


In [53]:
# let's move the network in GPU
net = net.to(device)

# define random batch of 10 elements
inp = torch.rand(10, 13).to(device)

# move the batch in GPU
inp = inp.to(device)

# get the output
out = net(inp)

# let's print the shape
print(' Input shape is', inp.shape)
print('Output shape is', out.shape)

 Input shape is torch.Size([10, 13])
Output shape is torch.Size([10, 1])


# Define validation routine

In [58]:
# create validation routine
def validate(net, dl):
    # get final score
    score = 0
    # set network in eval mode
    net.eval()
    # at the end of epoch, validate model
    for inp, gt in dl:
        # move batch to gpu
        inp = inp.to(device)
        gt = gt.to(device)
        # get output
        with torch.no_grad():
            out = net(inp)
        # compare with gt
        cur_score = F.l1_loss(out, gt)
        # append
        score += cur_score 
    # at the end, average over batches
    score /= len(dl)
    # set network in training mode
    net.train()
    # return score
    return score
        
        

# Train

In [59]:
import shutil

# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir={experiment_name}

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/bin/tensorboard", line 3, in <module>
    from tensorboard.main import run_main
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tensorboard/main.py", line 27, in <module>
    from tensorboard import default
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tensorboard/default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [60]:
# define optimizer
optimizer = torch.optim.Adam(params=net.parameters(), lr=learning_rate)

# define summary writer
writer = SummaryWriter(experiment_name)

# initialize iteration number
n_iter = 0

# define best validation value
best_val = None

# for each epoch
for cur_epoch in range(2500):

    net.train()

    # log current epoch
    writer.add_scalar("epoch", cur_epoch, n_iter)

    # for each batch
    for inp, gt in train_dl:
        # move batch to device
        inp = inp.to(device)
        gt = gt.to(device)

        # reset gradients
        optimizer.zero_grad()

        # forward pass
        out = net(inp)

        # compute loss
        loss = F.l1_loss(out, gt)

        # backward pass
        loss.backward()

        # update weights
        optimizer.step()

        # log training loss
        writer.add_scalar("loss", loss.item(), n_iter)
        n_iter += 1

    # validate model at the end of the epoch
    cur_val = validate(net, val_dl)

    # log validation score
    writer.add_scalar("val", cur_val, n_iter)

    # save best model
    if best_val is None or cur_val > best_val:
        best_val = cur_val
        torch.save({
            'net': net.state_dict(),
            'opt': optimizer.state_dict(),
            'epoch': cur_epoch
        }, experiment_name + '_best.pth')

    # always save last model
    torch.save({
        'net': net.state_dict(),
        'opt': optimizer.state_dict(),
        'epoch': cur_epoch
    }, experiment_name + '_last.pth')

RuntimeError: linear(): input and weight.T shapes cannot be multiplied (50x50 and 13x128)

# Test

In [ ]:
# create test dataset
test_ds =  Dataset('BostonHousingDataset/test.csv')

# create dataloader
test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 8
)

In [ ]:
# load best network
state = torch.load(experiment_name + '_best.pth')
net.load_state_dict(state['net'])

<All keys matched successfully>

In [ ]:
test_value = validate(net, test_dl).item()

In [ ]:
print(f'The model scored a MAE of {test_value:0.04f} over the testset.')

The model scored a MAE of 0.3966 over the testset.
